# Running ETL to Build the Document Corpus

This notebook walks through the process for setting up the corpus of Full Stack documents that the bot searches over.

In each case, we have to
- Extract data from its natural habitat, like YouTube or GitHub
- Transform it into a format that is useful for our purposes
- Load it into our database in that format

hence the acronym "ETL".

In [ ]:
import json
from pathlib import Path
import pprint
import docstore
from pymongo import ASCENDING
import os
from etl import markdown, pdfs, shared, videos
from dotenv import load_dotenv
from IPython.display import IFrame
from IPython.display import YouTubeVideo

load_dotenv(".env.dev")
os.environ["MODAL_ENVIRONMENT"] = "dev"
pp = pprint.PrettyPrinter(indent=2)

db = "fsdl-dev"
collection = "ask-fsdl"

## PDFs: arXiV Papers

```bash
modal run --env dev etl/pdfs.py \
    --json-path data/llm-papers.json \
    --db fsdl-dev --collection ask-fsdl
```

In [ ]:
col = docstore.get_database(db)[collection]
total = col.count_documents({})
print(f"total documents: {total}")

# Check for duplicate sha256 hashes (would indicate double-inserts)
pipeline = [
    {"$group": {
        "_id": "$metadata.sha256",
        "ids": {"$push": "$_id"},
        "n": {"$sum": 1},
    }},
    {"$match": {"n": {"$gt": 1}}},
]
dup = list(col.aggregate(pipeline))
print(f"duplicated hashes: {dup[0]['duplicated_hashes'] if dup else 0}")

# Sample a doc to confirm structure
d = col.find_one({})
if d:
    print("sample source:", d["metadata"].get("source"))
    print("sample title: ", d["metadata"].get("title"))
    print("sample chars: ", len(d.get("text", "")))

# Remove existing duplicates first, keeping the earliest _id per hash.
to_delete = []
for doc in col.aggregate(pipeline):
    to_delete.extend(doc["ids"][1:])

if to_delete:
    r = col.delete_many({"_id": {"$in": to_delete}})
    print(f"deleted {r.deleted_count} duplicates")

print(f"after cleanup: {col.count_documents({})}")

# Now add the unique index. Fails if any duplicate remains.
col.create_index("metadata.sha256", unique=True)
print("unique index on metadata.sha256 created")

In [ ]:
papers_path = Path("data") / "llm-papers.json"
with open(papers_path) as f:
    pdf_infos = json.load(f)

print(f"total entries: {len(pdf_infos)}")
print()
print("first entry:")
print(json.dumps(pdf_infos[0], indent=2))

In [ ]:
async with pdfs.app.run():
    paper_data = []
    async for item in pdfs.get_pdf_url.map.aio(
        pdf_infos[::25],
        return_exceptions=True,
    ):
        paper_data.append(item)

    batches = []
    async for batch in pdfs.extract_pdf.map.aio(
        paper_data,
        return_exceptions=True,
    ):
        batches.append(batch)

documents = shared.unchunk(batches)
print(f"{len(documents)} documents from {len(paper_data)} papers")

In [ ]:
pp.pprint(documents[0]["metadata"])

In [ ]:
IFrame(src=documents[0]["metadata"]["source"], width=800, height=400)

In [ ]:
async with shared.app.run():
    chunked_documents = shared.chunk_into(documents, 10)

    results = []
    async for result in shared.add_to_document_db.map.aio(
        chunked_documents,
        kwargs={"db": db, "collection": collection},
    ):
        results.append(result)

print(f"insert complete: {len(results)} chunks processed")

In [ ]:
query = {"metadata.source": {"$regex": r"arxiv\.org", "$options": "i"}}
projection = {"text": 0}
result = docstore.query_one(
    query,
    projection,
    db=db,
    collection=collection,
)

pp.pprint(result)

## Markdown Files: Lectures

```bash
modal run --env dev etl/markdown.py \
  --json-path data/lectures-2022.json \
  --db fsdl-dev --collection ask-fsdl
```

In [ ]:
markdown_path = Path("data") / "lectures-2022.json"   # data/lectures-2022.json

with open(markdown_path) as f:
    markdown_corpus = json.load(f)

website_url, md_url = (
    markdown_corpus["website_url_base"],   # e.g. https://fullstackdeeplearning.com/course/2022
    markdown_corpus["md_url_base"],        # e.g. https://raw.githubusercontent.com/the-full-stack/website/main/docs/course/2022
)

lectures = markdown_corpus["lectures"]     # list of dicts, one per lecture
lectures[0]                                # show the first one

In [ ]:
async with markdown.app.run():
    documents = shared.unchunk(
        [
            batch
            async for batch in markdown.to_documents.map.aio(
                lectures,
                kwargs={"website_url": website_url, "md_url": md_url},
                return_exceptions=True,
            )
        ]
    )

print(f"{len(documents)} documents from {len(lectures)} lectures")

In [ ]:
pp.pprint(documents[1]["metadata"])

In [ ]:
IFrame(src=documents[1]["metadata"]["source"], width=800, height=400)

In [ ]:
async with shared.app.run():
    chunked_documents = shared.chunk_into(documents, 10)

    results = [
        r
        async for r in shared.add_to_document_db.map.aio(
            chunked_documents,
            kwargs={"db": db, "collection": collection},
        )
    ]

print(f"insert complete: {len(results)} chunks processed")

In [ ]:
query = {"metadata.source": {"$regex": "lecture", "$options": "i"}}
projection = {"text": 0}
result = docstore.query_one(
    query,
    projection,
    db=db,
    collection=collection,
)

pp.pprint(result)

## Videos: YouTube Transcripts

```bash
modal run --env dev etl/videos.py \
    --json-path data/videos.json \
    --db fsdl-dev --collection ask-fsdl
```

In [ ]:
videos_path = Path("data") / "videos.json"

with open(videos_path) as f:
    video_infos = json.load(f)

video_ids = [video["id"] for video in video_infos]

video_infos[0]

In [ ]:
async with videos.app.run():
    raw = [
        batch
        async for batch in videos.extract_subtitles.map.aio(
            video_infos[-3:],
            return_exceptions=True,
        )
    ]

    documents = shared.unchunk(raw)

print(f"{len(documents)} documents from {len(raw)} videos")

In [ ]:
pp.pprint(documents[1]["metadata"])

In [ ]:
id_str, time_str = documents[1]["metadata"]["source"].split("?v=")[-1].split("&t=")
YouTubeVideo(id_str, start=int(time_str.strip("s")), width=800, height=400)

In [ ]:
async with shared.app.run():
    chunked_documents = shared.chunk_into(documents, 10)

    results = [
        r
        async for r in shared.add_to_document_db.map.aio(
            chunked_documents,
            kwargs={"db": db, "collection": collection},
        )
    ]

print(f"insert complete: {len(results)} chunks processed")

In [ ]:
query = {"metadata.source": {"$regex": "youtube", "$options": "i"}}
projection = {"text": 0}

result = docstore.query_one(
    query,
    projection,
    db=db,
    collection=collection,
)

pp.pprint(result)